# 15. 에이전트 평가

하네스 변경은 여러 태스크를 같은 기준으로 평가해야 판단할 수 있다. 결과 정확도와 실행 궤적은 별도로 측정한다.

### 공통 설정

`1`과 같은 블록이다. 노트북끼리 코드를 공유하지 않는다.

In [1]:
import json
import os
import time
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv(".env", usecwd=True), override=True)
MODEL = os.environ["OPENAI_DEFAULT_MODEL"]

client = OpenAI()

PRICES = {MODEL: (2.50, 15.00)}
USAGE = []


def keep(resp):
    USAGE.append((resp.model, resp.usage))
    return resp

In [2]:
CORPUS = Path("../../../../../data/corpus/graph")


def grep_docs(pattern):
    return [{"title": f.stem, "count": f.read_text(encoding="utf-8").count(pattern)}
            for f in sorted(CORPUS.glob("*.md")) if pattern in f.read_text(encoding="utf-8")]

### 코퍼스 기반 태스크셋

문항 유형은 사람이 정하고 정답은 코퍼스에서 코드로 생성한다.
집계, 최다 문서, 회상 유형을 섞되 모두 도구 사용을 요구한다. 회상으로 정답을 맞혀도 코퍼스를 확인하지 않았다면 궤적 실패다.

In [3]:
COUNT_WORDS = ["한국독립당", "의열단", "광복군", "신민회", "임시의정원",
               "독립신문", "충칭", "하와이", "대한인국민회", "흥사단"]
TOP_WORDS = ["한국독립당", "의열단", "광복군", "신민회", "임시의정원"]
RECALL = [("강우규가 1919년에 저격하려 한 인물은 누구인가", "사이토"),
          ("윤봉길이 1932년 의거를 일으킨 장소는 어디인가", "훙커우"),
          ("이봉창이 1932년에 폭탄을 던진 도시는 어디인가", "도쿄"),
          ("안중근이 1909년에 사살한 인물은 누구인가", "이토"),
          ("김구의 호는 무엇인가", "백범")]

집계와 최다 문항의 정답은 `grep_docs`로 만든다.

In [4]:
TASKS = []
for w in COUNT_WORDS:
    TASKS.append({"id": f"c-{w}", "유형": "집계", "도구필요": True,
                  "q": f"'{w}' 이(가) 나오는 문서가 몇 편인지 세고 숫자로 답한다",
                  "정답": [str(len(grep_docs(w)))]})
for w in TOP_WORDS:
    hits = grep_docs(w)
    top = max(d["count"] for d in hits)
    TASKS.append({"id": f"t-{w}", "유형": "최다", "도구필요": True,
                  "q": f"'{w}' 이(가) 가장 많이 나오는 문서의 제목을 답한다",
                  "정답": [d["title"] for d in hits if d["count"] == top]})

회상 문항의 정답은 사람이 적었다.

In [5]:
for q, a in RECALL:
    TASKS.append({"id": f"r-{a}", "유형": "회상", "도구필요": True, "q": q, "정답": [a]})

print(f"태스크 {len(TASKS)}개", {t: sum(1 for x in TASKS if x['유형'] == t) for t in ("집계", "최다", "회상")})

태스크 20개 {'집계': 10, '최다': 5, '회상': 5}


### 답과 궤적 기록

도구와 설정은 `8`과 같다.

In [6]:
TOOLS = [{"type": "function", "name": "grep_docs", "strict": True,
          "description": "낱말이 든 문서의 제목과 등장 횟수를 돌려준다.",
          "parameters": {"type": "object",
                         "properties": {"pattern": {"type": "string", "description": "찾을 낱말"}},
                         "required": ["pattern"], "additionalProperties": False}}]

CONFIG = {"instructions": "도구로 확인한 것만 답한다.", "max_turns": 5,
          "tool_error": "error: {e}"}

사후 궤적 평가를 위해 답과 도구 호출 기록을 함께 남긴다.

In [7]:
def run(task):
    messages = [{"role": "user", "content": task}]
    tools_used, errors, t0, used = [], 0, time.perf_counter(), 0
    for turn in range(1, CONFIG["max_turns"] + 1):
        r = keep(client.responses.create(model=MODEL, instructions=CONFIG["instructions"],
                                         input=messages, tools=TOOLS, store=False))
        used += r.usage.total_tokens
        calls = [i for i in r.output if i.type == "function_call"]
        if not calls:
            return {"답": r.output_text, "도구": tools_used, "턴": turn, "오류": errors,
                    "토큰": used, "초": round(time.perf_counter() - t0, 1)}
        messages += [i.model_dump() for i in r.output]
        for c in calls:
            tools_used.append(c.name)
            try:
                body = json.dumps(grep_docs(**json.loads(c.arguments)), ensure_ascii=False)
            except Exception as e:
                body = CONFIG["tool_error"].format(e=e)
                errors += 1
            messages.append({"type": "function_call_output", "call_id": c.call_id, "output": body})
    return {"답": None, "도구": tools_used, "턴": CONFIG["max_turns"], "오류": errors,
            "토큰": used, "초": round(time.perf_counter() - t0, 1)}

한 문항을 돌려 기록의 모양을 본다.

In [8]:
rec = run(TASKS[0]["q"])
print(rec)

{'답': '16', '도구': ['grep_docs'], '턴': 2, '오류': 0, '토큰': 509, '초': 3.2}


### 결과와 궤적 채점

결과 채점은 정답 포함 여부를 본다. **궤적 채점**은 도구 사용, 오류, 턴 수를 평가한다.

In [9]:
def grade_result(task, rec):    # 결과 채점
    return any(a in (rec["답"] or "") for a in task["정답"])


def grade_trajectory(task, rec):    # 궤적 채점
    if task["도구필요"] and not rec["도구"]:
        return False, "도구를 쓰지 않았다"
    if rec["오류"]:
        return False, f"도구 오류 {rec['오류']}건"
    if rec["턴"] > 3:
        return False, f"턴 {rec['턴']}"
    return True, "정상"

위의 기록을 채점한다.

In [10]:
print("결과", grade_result(TASKS[0], rec), "  궤적", grade_trajectory(TASKS[0], rec))

결과 True   궤적 (True, '정상')


태스크셋 전체를 돌려 문항마다 기록과 두 점수를 한 줄로 모은다.

In [11]:
def evaluate(tasks):
    rows = []
    for t in tasks:
        rec = run(t["q"])
        ok = grade_result(t, rec)
        traj, why = grade_trajectory(t, rec)
        rows.append({**t, **rec, "결과": ok, "궤적": traj, "사유": why})
    return rows

In [12]:
rows = evaluate(TASKS)
print(f"기본 지시   결과 {sum(r['결과'] for r in rows)}/{len(rows)}"
      f"   궤적 {sum(r['궤적'] for r in rows)}/{len(rows)}")

기본 지시   결과 17/20   궤적 20/20


### 지시 변경 비교

태스크셋, 도구, 모델을 고정하고 지시만 바꿔 두 점수의 변화를 비교한다.
</br>궤적을 쓰지 않은 경우, 도구없이도 답변할 수 있을거라 판단하여 쓰지 않음

In [ ]:
saved = CONFIG["instructions"]
CONFIG["instructions"] = "아는 대로 빠르게 답한다."
lazy = evaluate(TASKS)
CONFIG["instructions"] = saved

print(f"빠르게 답함  결과 {sum(r['결과'] for r in lazy)}/{len(lazy)}"   # 도구 활용없이도 이미 있는 지식으로 답할 것이라 예측
      f"   궤적 {sum(r['궤적'] for r in lazy)}/{len(lazy)}")
print(f"도구를 쓴 문항  기본 {sum(bool(r['도구']) for r in rows)}"
      f"   빠르게 {sum(bool(r['도구']) for r in lazy)}")

빠르게 답함  결과 20/20   궤적 15/20
도구를 쓴 문항  기본 20   빠르게 15


### 결과와 궤적 조합

두 점수를 사분면으로 나눈다.

In [14]:
def quadrants(rs, label):
    box = {(True, True): [], (True, False): [], (False, True): [], (False, False): []}
    for r in rs:
        box[(r["결과"], r["궤적"])].append(r)
    print(f"{label}  둘 다 맞음 {len(box[(True, True)]):2d}"
          f"   운 좋게 맞음 {len(box[(True, False)]):2d}"
          f"   제대로 하고 틀림 {len(box[(False, True)]):2d}"
          f"   둘 다 틀림 {len(box[(False, False)]):2d}")
    return box

In [15]:
base_box = quadrants(rows, "기본 지시  ")
lazy_box = quadrants(lazy, "빠르게 답함")

기본 지시    둘 다 맞음 17   운 좋게 맞음  0   제대로 하고 틀림  3   둘 다 틀림  0
빠르게 답함  둘 다 맞음 15   운 좋게 맞음  5   제대로 하고 틀림  0   둘 다 틀림  0


운 좋게 맞은 문항을 몇 개 본다.

In [16]:
for r in lazy_box[(True, False)][:3]:
    print(f"[{r['id']}] {r['q'][:34]}")
    print(f"  답 {str(r['답'])[:44]}   정답 {r['정답'][:2]}   사유 {r['사유']}")

[r-사이토] 강우규가 1919년에 저격하려 한 인물은 누구인가
  답 1919년에 강우규가 저격하려 한 인물은 **사이토 마코토(齋藤實)**, 당시    정답 ['사이토']   사유 도구를 쓰지 않았다
[r-훙커우] 윤봉길이 1932년 의거를 일으킨 장소는 어디인가
  답 윤봉길이 1932년 의거를 일으킨 장소는 **중국 상하이 훙커우 공원**입니다.   정답 ['훙커우']   사유 도구를 쓰지 않았다
[r-도쿄] 이봉창이 1932년에 폭탄을 던진 도시는 어디인가
  답 이봉창이 1932년에 폭탄을 던진 도시는 **도쿄(東京)**입니다.   정답 ['도쿄']   사유 도구를 쓰지 않았다


### 결과만 맞은 경우

지시 변경 후 결과 점수는 올랐지만 **궤적 점수는 내려갈 수 있다.** 위키백과 기반 코퍼스는 모델이 내용을 기억할 수 있어, 검색 없이도 정답을 맞힐 수 있기 때문이다.

결과만 평가하면 코퍼스가 낯선 문서로 바뀔 때 실패할 설정을 채택할 수 있다.

### 평가 지표

성공률만으로는 비용과 실행 효율을 구분할 수 없다.

In [17]:
n = len(rows)
spent = sum(r["토큰"] for r in rows) / 1e6 * PRICES[MODEL][0]
need_tool = [r for r in rows if r["도구필요"]]

print(f"결과 성공률      {sum(r['결과'] for r in rows) / n:.0%}")
print(f"궤적 정상률      {sum(r['궤적'] for r in rows) / n:.0%}")
print(f"도구 선택 정확도 {sum(bool(r['도구']) for r in need_tool) / len(need_tool):.0%}"
      f"   (도구가 필요한 {len(need_tool)}문항 기준)")
print(f"평균 턴          {sum(r['턴'] for r in rows) / n:.1f}")
print(f"비용당 성공      {sum(r['결과'] for r in rows) / max(spent, 1e-9):.0f}건/$1"
      f"   (이번 실행 ${spent:.4f})")

결과 성공률      85%
궤적 정상률      100%
도구 선택 정확도 100%   (도구가 필요한 20문항 기준)
평균 턴          2.2
비용당 성공      577건/$1   (이번 실행 $0.0295)


### 회귀 스위트

태스크셋과 현재 점수를 기준선으로 파일에 저장한다.

In [18]:
SUITE = Path("taskset.json")
BASE = {"결과": sum(r["결과"] for r in rows) / n, "궤적": sum(r["궤적"] for r in rows) / n}
SUITE.write_text(json.dumps({"tasks": TASKS, "baseline": BASE}, ensure_ascii=False, indent=1),
                 encoding="utf-8")
print("기준선", {k: f"{v:.0%}" for k, v in BASE.items()})

기준선 {'결과': '85%', '궤적': '100%'}


이후 하네스 변경은 기준선에서 `slack`보다 떨어지면 회귀로 본다.

In [ ]:
def check(rows_now, baseline, slack=0.05):  #0.05보다 크게 떨어지면 회귀
    bad = []
    for key in baseline:
        now = sum(r[key] for r in rows_now) / len(rows_now)
        if now < baseline[key] - slack:
            bad.append(f"{key} {baseline[key]:.0%} -> {now:.0%}")
    return bad

In [21]:
print("기본 지시 검사", check(rows, BASE) or "통과")
print("빠르게 답함 검사", check(lazy, BASE) or "통과")

기본 지시 검사 통과
빠르게 답함 검사 ['궤적 100% -> 75%']


### 평가 범위와 한계

서술 품질은 **사람 라벨** 없이 평가하지 않는다. 궤적 평가는 실패 원인까지 설명하지 않는다.(도구 call 여부) 변경 효과는 별도의 재평가가 필요하다.

태스크가 20개뿐이므로 한 문항이 점수를 5%p 바꾼다.(태스크 숫자가 많아야 => 수백개는 있어야하고 3번은 돌려야) 평균과 함께 결과가 뒤집힌 문항을 확인해야 한다.

### 비용 확인

In [22]:
total = 0.0
for name, u in USAGE:
    p = next((PRICES[k] for k in sorted(PRICES, key=len, reverse=True) if name.startswith(k)), None)
    if p:
        total += u.input_tokens / 1e6 * p[0] + u.output_tokens / 1e6 * p[1]

print(f"호출 {len(USAGE)}회")
print(f"입력 {sum(u.input_tokens for _, u in USAGE):,} 토큰   출력 {sum(u.output_tokens for _, u in USAGE):,} 토큰")
print(f"누적 ${total:.4f}")

호출 82회
입력 17,673 토큰   출력 1,631 토큰
누적 $0.0686
